In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import dask
import zarr
import xarray as xr
import cftime
import os

In [ ]:
llc_path = '/orcd/data/abodner/002/cody/LLC_patch/LLC4320_face1_i2880-3600_j720-1440.zarr'
llc_patch_full = xr.open_dataset(llc_path, consolidated=True)

# epoch 1
# epochs=1, steps=1, vars=all, loss=mae gradient, norm=group_norm_32, padding='constant,' pred_residual=true,
emulator_1_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-24-eval:Samudra_LLC:config_tests_experiment_6/predictions_4d.zarr'
emulator_1_patch_full = xr.open_dataset(emulator_1_path, consolidated=True) 

# epoch 2
# epochs=2, steps=1, vars=all, loss=mae gradient, norm=group_norm_32, padding='constant,' pred_residual=true, face=1, i=[2880:3600), j=[720:1440)
emulator_2_path = '/orcd/data/abodner/002/cody/inference_patch/multi_epoch_tests_4-25-26/2026-04-24-eval:Samudra_LLC:config_tests_experiment_6_epoch2/predictions_4d.zarr'
emulator_2_patch_full = xr.open_dataset(emulator_2_path, consolidated=True) 

# epoch 2 OOD
# # epochs=2, steps=1, vars=all, loss=mae gradient, norm=group_norm_32, padding='constant,' pred_residual=true, OOD, face=1, i=[1440:2160), j=[1440:2160)
# emulator_3_path = '/orcd/data/abodner/002/cody/inference_patch/multi_epoch_tests_4-25-26/2026-04-24-eval:Samudra_LLC:config_tests_experiment_6_epoch2_OOD/predictions_4d.zarr'
# emulator_3_patch_full = xr.open_dataset(emulator_3_path, consolidated=True) 



# # EXPERIMENT 1
# #epochs=1, steps=1, vars=all, loss=mse_diff_weighted, norm=syncbatchnorm, padding='constant,' pred_residual=true,
# emulator_1_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-23-eval:Samudra_LLC:config_tests_experiment_1/predictions_4d.zarr'
# emulator_1_patch_full = xr.open_dataset(emulator_1_path, consolidated=True) 

# # EXPERIMENT 2
# # epochs=1, steps=1, vars=all, loss=mse_diff_weighted, norm=group_norm_32, padding='constant,' pred_residual=true,
# emulator_2_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-23-eval:Samudra_LLC:config_tests_experiment_2/predictions_4d.zarr'
# emulator_2_patch_full = xr.open_dataset(emulator_2_path, consolidated=True) 

# # EXPERIMENT 3
# # epochs=1, steps=1, vars=all, loss=mse_diff_weighted + dynamically weighted loss, norm=syncbatchnorm, padding='constant,' pred_residual=true,
# # emulator_3_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-23-eval:Samudra_LLC:config_tests_experiment_3/predictions_4d.zarr'
# # emulator_3_patch_full = xr.open_dataset(emulator_3_path, consolidated=True) 

# # EXPERIMENT 4
# # epochs=1, steps=1, vars=all, loss=mse_diff_weighted + dynamically weighted loss, norm=syncbatchnorm, padding='halo_sponge,' pred_residual=true
# #emulator_4_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-23-eval:Samudra_LLC:config_tests_experiment_4/predictions_4d.zarr'
# #emulator_4_patch_full = xr.open_dataset(emulator_4_path, consolidated=True) 

# # EXPERIMENT 5
# # epochs=1, steps=1, vars=all, loss=mse_diff_weighted + weighted loss [U,V = 1.0, Theta, Salt, Eta = 1.5 , norm=group_norm_32, padding='constant,' pred_residual=true,
# emulator_3_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-24-eval:Samudra_LLC:config_tests_experiment_5/predictions_4d.zarr'
# emulator_3_patch_full = xr.open_dataset(emulator_3_path, consolidated=True) 

# # EXPERIMENT 6
# # epochs=1, steps=1, vars=all, loss=mae gradient, norm=group_norm_32, padding='constant,' pred_residual=true,
# emulator_4_path = '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-24-eval:Samudra_LLC:config_tests_experiment_6/predictions_4d.zarr'
# emulator_4_patch_full = xr.open_dataset(emulator_4_path, consolidated=True) 

In [2]:
# ============== LOAD LLC ==============
llc_path = '/orcd/data/abodner/002/cody/LLC_patch/LLC4320_face1_i2880-3600_j720-1440.zarr'
llc_patch_full = xr.open_dataset(llc_path, consolidated=True)

# ============== LOAD EMULATORS ==============
# Add or remove entries here to control how many emulators are compared
emulator_configs = [
    {
        'name': 'Emulator 1',
        'key': 'emulator_1',
        'path': '/orcd/data/abodner/002/cody/inference_patch/config_tests_4-23-26/2026-04-24-eval:Samudra_LLC:config_tests_experiment_6/predictions_4d.zarr',
        'desc': 'epochs=1, steps=1, vars=all, loss=mae gradient, norm=group_norm_32, padding=constant, pred_residual=true'
    },
    {
        'name': 'Emulator 2',
        'key': 'emulator_2',
        'path': '/orcd/data/abodner/002/cody/inference_patch/multi_epoch_tests_4-25-26/2026-04-24-eval:Samudra_LLC:config_tests_experiment_6_epoch2/predictions_4d.zarr',
        'desc': 'epochs=2, steps=1, vars=all, loss=mae gradient, norm=group_norm_32, padding=constant, pred_residual=true'
    },
    # {
    #     'name': 'Emulator 3',
    #     'key': 'emulator_3',
    #     'path': '/orcd/data/abodner/002/cody/inference_patch/multi_epoch_tests_4-25-26/2026-04-24-eval:Samudra_LLC:config_tests_experiment_6_epoch2_OOD/predictions_4d.zarr',
    #     'desc': 'epochs=2, steps=1, OOD, face=1, i=[1440:2160), j=[1440:2160)'
    # },
    # Add more emulators here as needed:
    # {
    #     'name': 'Emulator 4',
    #     'key': 'emulator_4',
    #     'path': '...',
    #     'desc': '...'
    # },
]

# ============== OPEN EMULATOR DATASETS ==============
emulator_patches_raw = {}
for cfg in emulator_configs:
    emulator_patches_raw[cfg['key']] = xr.open_dataset(cfg['path'], consolidated=True)
    print(f"Loaded {cfg['name']}: {cfg['desc']}")

# ============== TIME MATCHING ==============
# Use first emulator's times as reference
first_key = emulator_configs[0]['key']
emulator_times = emulator_patches_raw[first_key].time.values
matching_times = pd.DatetimeIndex([
    pd.Timestamp(t.year, t.month, t.day, t.hour, t.minute, t.second)
    for t in emulator_times
])

llc_patch = llc_patch_full.sel(time=matching_times)
print(f"LLC subset to {len(matching_times)} matching times")

# ============== PORT GRID VARS & BUILD UNIFIED STRUCTURE ==============
grid_vars = ['XC', 'YC', 'rA', 'Z']

emulator_patches = {}
for cfg in emulator_configs:
    patch = emulator_patches_raw[cfg['key']]
    for gv in grid_vars:
        patch[gv] = llc_patch[gv]
    emulator_patches[cfg['key']] = patch

# ============== UNIFIED REFERENCE LISTS ==============
# This is what all plotting scripts will use
emulator_info = [(cfg['name'], cfg['key']) for cfg in emulator_configs]
n_emulators = len(emulator_info)

all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

print(f"\n=== Setup complete: LLC + {n_emulators} emulators ===")
for name, key in emulator_info:
    print(f"  {name} ({key})")

Loaded Emulator 1: epochs=1, steps=1, vars=all, loss=mae gradient, norm=group_norm_32, padding=constant, pred_residual=true
Loaded Emulator 2: epochs=2, steps=1, vars=all, loss=mae gradient, norm=group_norm_32, padding=constant, pred_residual=true
LLC subset to 10 matching times

=== Setup complete: LLC + 2 emulators ===
  Emulator 1 (emulator_1)
  Emulator 2 (emulator_2)


In [3]:
def format_time(t_val):
    """Format a time value to DD/MM/YYYY:HH regardless of cftime or datetime64."""
    try:
        return f"{t_val.day:02d}/{t_val.month:02d}/{t_val.year}:{t_val.hour:02d}h"
    except AttributeError:
        t_pd = pd.Timestamp(t_val)
        return f"{t_pd.day:02d}/{t_pd.month:02d}/{t_pd.year}:{t_pd.hour:02d}h"

# surface field and field difference

In [4]:
# ============== SET VARIABLES HERE ==============
prog_vars = ['Theta', 'Salt', 'U', 'V']
colormaps = {'Theta': 'Spectral_r', 'Salt': 'viridis', 'U': 'bwr', 'V': 'bwr'}
# ================================================
for var in prog_vars:
    print(f"Generating plots for {var}...")
    
    os.makedirs(f'figs/prognostic_var_comparison/{var}', exist_ok=True)
    
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    nrows = len(time_indices)
    ncols_fields = 1 + n_emulators  # LLC + emulators
    ncols_diff = n_emulators         # emulators only
    
    cmap = colormaps[var]
    
    # ==================== PLOT 1: Surface fields ====================
    fig, axes = plt.subplots(nrows, ncols_fields, figsize=(3.6*ncols_fields, 3*nrows), dpi=200)
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    if ncols_fields == 1:
        axes = axes.reshape(-1, 1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        # Collect all surface fields
        fields = [llc_patch.isel(time=t, k=0)[var]]
        for emu_name, emu_key in emulator_info:
            fields.append(emulator_patches[emu_key].isel(time=t, k=0)[var])
        
        vmin = np.min([f.values.min() for f in fields])
        vmax = np.max([f.values.max() for f in fields])
        
        labels = ['LLC'] + [name for name, _ in emulator_info]
        
        for col, (field, label) in enumerate(zip(fields, labels)):
            ax = axes[row, col]
            cf = ax.contourf(field.coords.get('i', np.arange(field.shape[-1])),
                             field.coords.get('j', np.arange(field.shape[-2])), field,
                             cmap=cmap, vmin=vmin, vmax=vmax, levels=30)
            ax.set_title(f'{label} {var} {time_str}', fontsize=8)
            plt.colorbar(cf, ax=ax)
    
    plt.tight_layout()
    plt.savefig(f'figs/prognostic_var_comparison/{var}/surface_{var}_fields.png')
    plt.close()
    
    # ==================== PLOT 2: Difference fields ====================
    fig, axes = plt.subplots(nrows, ncols_diff, figsize=(4*ncols_diff, 3*nrows), dpi=200)
    
    if nrows == 1 and ncols_diff > 1:
        axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols_diff == 1:
        axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols_diff == 1:
        axes = axes.reshape(1, 1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        llc_vis = llc_patch.isel(time=t, k=0)[var]
        
        diffs = []
        for emu_name, emu_key in emulator_info:
            emu_vis = emulator_patches[emu_key].isel(time=t, k=0)[var]
            diffs.append(llc_vis.values - emu_vis.values)
        
        abs_max = np.max([np.abs(d).max() for d in diffs])
        vmin_d, vmax_d = -abs_max, abs_max
        
        row_axes = [axes[row, col] for col in range(ncols_diff)]
        
        for col, ((emu_name, _), diff) in enumerate(zip(emulator_info, diffs)):
            ax = row_axes[col]
            cf = ax.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])),
                             llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), diff,
                             cmap="bwr", vmin=vmin_d, vmax=vmax_d, levels=30)
            short_name = emu_name.replace('Emulator ', 'Em')
            ax.set_title(f'LLC - {short_name} {var} {time_str}', fontsize=8)
        
        fig.colorbar(cf, ax=row_axes, orientation='vertical',
                     fraction=0.046, pad=0.04)
    
    plt.savefig(f'figs/prognostic_var_comparison/{var}/surface_{var}_differences.png')
    plt.close()
    
    print(f"✓ Saved plots for {var}")

Generating plots for Theta...
✓ Saved plots for Theta
Generating plots for Salt...
✓ Saved plots for Salt
Generating plots for U...
✓ Saved plots for U
Generating plots for V...
✓ Saved plots for V


# Gradients

In [5]:
grad_vars = ['Theta', 'Salt', 'U', 'V']

for var in grad_vars:
    grad_name = f'grad_{var}'
    print(f"Computing {grad_name}...")
    
    for patch_name, patch in all_patches.items():
        data = patch[var].values  # (time, k, j, i)
        
        dx = np.sqrt(patch['rA'].values)  # (j, i) in meters
        dy = dx.copy()
        
        d_di = (np.roll(data, -1, axis=3) - np.roll(data, 1, axis=3)) / (2 * dx[np.newaxis, np.newaxis, :, :])
        d_dj = (np.roll(data, -1, axis=2) - np.roll(data, 1, axis=2)) / (2 * dy[np.newaxis, np.newaxis, :, :])
        
        grad_mag = np.sqrt(d_di**2 + d_dj**2)
        
        patch[grad_name] = (('time', 'k', 'j', 'i'), grad_mag)
        print(f"  ✓ {patch_name} {grad_name}: {grad_mag.shape}")

print("Done computing gradients!")

Computing grad_Theta...
  ✓ llc grad_Theta: (10, 51, 720, 720)
  ✓ emulator_1 grad_Theta: (10, 51, 720, 720)
  ✓ emulator_2 grad_Theta: (10, 51, 720, 720)
Computing grad_Salt...
  ✓ llc grad_Salt: (10, 51, 720, 720)
  ✓ emulator_1 grad_Salt: (10, 51, 720, 720)
  ✓ emulator_2 grad_Salt: (10, 51, 720, 720)
Computing grad_U...
  ✓ llc grad_U: (10, 51, 720, 720)
  ✓ emulator_1 grad_U: (10, 51, 720, 720)
  ✓ emulator_2 grad_U: (10, 51, 720, 720)
Computing grad_V...
  ✓ llc grad_V: (10, 51, 720, 720)
  ✓ emulator_1 grad_V: (10, 51, 720, 720)
  ✓ emulator_2 grad_V: (10, 51, 720, 720)
Done computing gradients!


In [6]:
gradient_masks = {}

for patch_name, patch in all_patches.items():
    gradient_masks[patch_name] = {}
    
    for var in grad_vars:
        grad_name = f'grad_{var}'
        grad_data = patch[grad_name].values  # (time, k, j, i)
        
        n_times, n_depths = grad_data.shape[0], grad_data.shape[1]
        mask = np.zeros_like(grad_data, dtype=bool)
        
        for t in range(n_times):
            for k in range(n_depths):
                field = grad_data[t, k]
                threshold = np.nanpercentile(field, 97.5)
                mask[t, k] = field >= threshold
        
        gradient_masks[patch_name][var] = mask
        print(f"✓ {patch_name} {var}: {mask.sum()} high-gradient pixels ({mask.sum() / mask.size * 100:.1f}%)")

print("Done creating gradient masks!")

✓ llc Theta: 6609273 high-gradient pixels (2.5%)
✓ llc Salt: 6609268 high-gradient pixels (2.5%)
✓ llc U: 6609157 high-gradient pixels (2.5%)
✓ llc V: 6609224 high-gradient pixels (2.5%)
✓ emulator_1 Theta: 6609641 high-gradient pixels (2.5%)
✓ emulator_1 Salt: 6609631 high-gradient pixels (2.5%)
✓ emulator_1 U: 6609642 high-gradient pixels (2.5%)
✓ emulator_1 V: 6609643 high-gradient pixels (2.5%)
✓ emulator_2 Theta: 6609636 high-gradient pixels (2.5%)
✓ emulator_2 Salt: 6609630 high-gradient pixels (2.5%)
✓ emulator_2 U: 6609645 high-gradient pixels (2.5%)
✓ emulator_2 V: 6609645 high-gradient pixels (2.5%)
Done creating gradient masks!


In [7]:
for var in grad_vars:
    print(f"Generating gradient drift figure for {var}...")
    
    os.makedirs(f'figs/gradients/{var}', exist_ok=True)
    
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    nrows = len(time_indices)
    ncols = n_emulators
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.5*nrows), dpi=150)
    
    if nrows == 1 and ncols > 1:
        axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols == 1:
        axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols == 1:
        axes = axes.reshape(1, 1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        llc_mask_surface = gradient_masks['llc'][var][t, 0]  # (j, i)
        
        for col, (emu_name, emu_key) in enumerate(emulator_info):
            ax = axes[row, col]
            
            emu_field = all_patches[emu_key].isel(time=t, k=0)[var].values
            emu_mask_surface = gradient_masks[emu_key][var][t, 0]
            
            ax.imshow(emu_field, cmap='Greys', aspect='auto', origin='lower')
            
            overlap_mask = llc_mask_surface & emu_mask_surface
            llc_only_mask = llc_mask_surface & ~emu_mask_surface
            emu_only_mask = emu_mask_surface & ~llc_mask_surface
            
            llc_j, llc_i = np.where(llc_only_mask)
            emu_j, emu_i = np.where(emu_only_mask)
            ovl_j, ovl_i = np.where(overlap_mask)
            
            ax.scatter(llc_i, llc_j, c='red', s=1, alpha=0.5, label='LLC top 2.5%', rasterized=True)
            ax.scatter(emu_i, emu_j, c='green', s=1, alpha=0.5, label='Emu top 2.5%', rasterized=True)
            ax.scatter(ovl_i, ovl_j, c='yellow', s=1, alpha=0.7, label='Overlap', rasterized=True)
            
            n_overlap = np.sum(overlap_mask)
            
            ax.set_title(f'{emu_name} {var} {time_str} overlap={n_overlap}', fontsize=8)
            ax.tick_params(labelsize=6)
            
            if row == 0 and col == 0:
                ax.legend(fontsize=5, loc='upper right', markerscale=5)
    
    plt.tight_layout()
    plt.savefig(f'figs/gradients/{var}/surface_gradient_drift.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved gradient drift figure for {var}")

print("Done with gradient drift figures!")

Generating gradient drift figure for Theta...
✓ Saved gradient drift figure for Theta
Generating gradient drift figure for Salt...
✓ Saved gradient drift figure for Salt
Generating gradient drift figure for U...
✓ Saved gradient drift figure for U
Generating gradient drift figure for V...
✓ Saved gradient drift figure for V
Done with gradient drift figures!


# Error vs depth plots

In [8]:
depth_vars = ['Theta', 'Salt', 'U', 'V']
ref_lines = {
    'Theta': [0.5, 1.0],
    'Salt': [0.06, 0.12],
    'U': [0.075, 0.15],
    'V': [0.075, 0.15]
}

for var in depth_vars:
    print(f"Generating augmented depth error plots for {var}...")
    
    os.makedirs(f'figs/prognostic_var_comparison/{var}', exist_ok=True)
    
    n_depths = llc_patch.sizes['k']
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    
    nrows = len(time_indices)
    ncols = n_emulators
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows), dpi=150)
    
    if nrows == 1 and ncols > 1:
        axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols == 1:
        axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols == 1:
        axes = axes.reshape(1, 1)
    
    depths = np.arange(n_depths)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        llc_data = llc_patch.isel(time=t)[var].values  # (k, j, i)
        llc_grad_mask = gradient_masks['llc'][var][t]   # (k, j, i)
        
        # First pass: compute all errors for shared xlim
        row_mean_errors = []
        row_median_errors = []
        row_hg_mean_errors = []
        
        for emu_name, emu_key in emulator_info:
            emu_data = emulator_patches[emu_key].isel(time=t)[var].values
            diff = np.abs(llc_data - emu_data)
            
            diff_flat = diff.reshape(n_depths, -1)
            mean_errors = np.nanmean(diff_flat, axis=1)
            median_errors = np.nanmedian(diff_flat, axis=1)
            
            hg_mean_errors = np.zeros(n_depths)
            for k in range(n_depths):
                hg_pixels = diff[k][llc_grad_mask[k]]
                if len(hg_pixels) > 0:
                    hg_mean_errors[k] = np.nanmean(hg_pixels)
                else:
                    hg_mean_errors[k] = np.nan
            
            row_mean_errors.append(mean_errors)
            row_median_errors.append(median_errors)
            row_hg_mean_errors.append(hg_mean_errors)
        
        all_errors = np.concatenate(row_mean_errors + row_median_errors + row_hg_mean_errors)
        xmin = 0
        xmax = np.nanmax(all_errors) * 1.05
        
        # Second pass: plot
        for col, (emu_name, _) in enumerate(emulator_info):
            ax = axes[row, col]
            
            ax.scatter(row_mean_errors[col], depths, color='blue', s=30, alpha=0.7, zorder=3)
            ax.plot(row_mean_errors[col], depths, color='blue', alpha=0.4, linewidth=1.5, label='Mean')
            
            ax.scatter(row_median_errors[col], depths, color='red', s=30, alpha=0.7, zorder=3)
            ax.plot(row_median_errors[col], depths, color='red', alpha=0.4, linewidth=1.5, label='Median')
            
            ax.scatter(row_hg_mean_errors[col], depths, color='green', s=30, alpha=0.7, zorder=3)
            ax.plot(row_hg_mean_errors[col], depths, color='green', alpha=0.4, linewidth=1.5, label='HG Mean')
            
            for ref_val in ref_lines[var]:
                ax.axvline(x=ref_val, color='black', linestyle='--', linewidth=1.5, alpha=0.5, zorder=2)
            
            ax.set_title(f'{emu_name} {var} {time_str}', fontsize=8)
            ax.set_xlabel('Abs Error', fontsize=7)
            ax.set_ylabel('Depth (k)', fontsize=7)
            ax.set_ylim(n_depths - 1, 0)
            ax.set_xlim(xmin, xmax)
            ax.grid(alpha=0.2)
            ax.tick_params(labelsize=6)
            
            if col == 0:
                ax.legend(fontsize=6, loc='lower right')
    
    plt.tight_layout()
    plt.savefig(f'figs/prognostic_var_comparison/{var}/depth_error_by_time.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved augmented depth error plots for {var}")

print("Done!")

Generating augmented depth error plots for Theta...
✓ Saved augmented depth error plots for Theta
Generating augmented depth error plots for Salt...
✓ Saved augmented depth error plots for Salt
Generating augmented depth error plots for U...
✓ Saved augmented depth error plots for U
Generating augmented depth error plots for V...
✓ Saved augmented depth error plots for V
Done!
